# Pendulum Q-Learning
**Algorithm:** Tabular Q-Learning  
**Training requirement:** 1,000 episodes  
**Checkpoint:** every 250 episodes  
**Resume:** automatically continues from the latest Q-table checkpoint

## 1. Setup
Install Gymnasium and plotting/video packages.

In [ ]:
%pip install -q \
    "gymnasium[classic-control]>=1.0" \
    "imageio[ffmpeg]" \
    "pandas" \
    "matplotlib"

## 2. Configuration
Set the seed, folders, training requirement, and evaluation episodes.

In [ ]:
import re
import random
import base64
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio.v2 as imageio

import gymnasium as gym

from IPython import display as ipythondisplay

SEED = 42
EPISODES = 1_000
EVAL_EPISODES = 100
CHECKPOINT_INTERVAL_EPISODES = 250
LOG_INTERVAL_EPISODES = 100

random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("as2_pendulum")
MODEL_DIR = ROOT / "models"
CHECKPOINT_DIR = ROOT / "checkpoints"
LOG_DIR = ROOT / "logs"
VIDEO_DIR = ROOT / "videos"

for folder in [
    MODEL_DIR,
    CHECKPOINT_DIR,
    LOG_DIR,
    VIDEO_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

## 3. Fixed Hyperparameters
Continuous states and torques are discretised before applying tabular Q-Learning.

In [ ]:
N_COS = 15
N_SIN = 15
N_VEL = 25
N_ACTIONS = 11

ALPHA = 0.12
GAMMA = 0.99
EPSILON_START = 1.0
EPSILON_END = 0.05
EPSILON_DECAY = 0.995

## 4. Environment and Discretisation
Pendulum has a continuous observation and action space. Both are discretised for a tabular TD control method.

In [ ]:
ENV_ID = "Pendulum-v1"

cos_edges = np.linspace(
    -1.0,
    1.0,
    N_COS - 1,
)

sin_edges = np.linspace(
    -1.0,
    1.0,
    N_SIN - 1,
)

vel_edges = np.linspace(
    -8.0,
    8.0,
    N_VEL - 1,
)

actions = np.linspace(
    -2.0,
    2.0,
    N_ACTIONS,
    dtype=np.float32,
)


def discretise(
    observation,
):
    c, s, v = observation

    return (
        int(
            np.digitize(
                np.clip(
                    c,
                    -1,
                    1,
                ),
                cos_edges,
            )
        ),
        int(
            np.digitize(
                np.clip(
                    s,
                    -1,
                    1,
                ),
                sin_edges,
            )
        ),
        int(
            np.digitize(
                np.clip(
                    v,
                    -8,
                    8,
                ),
                vel_edges,
            )
        ),
    )

## 5. Utilities
Utilities handle checkpoint discovery, CSV logging, evaluation, plotting, and video playback.

In [ ]:
Q_LOG_PATH = (
    LOG_DIR
    / "q_learning_training_log.csv"
)


def latest_q_checkpoint():
    files = list(
        CHECKPOINT_DIR.glob(
            "pendulum_q_"
            "*_episodes.npz"
        )
    )

    if not files:
        return None, 0

    def get_episode(path):
        match = re.search(
            r"_(\d+)_episodes\.npz$",
            path.name,
        )

        return (
            int(match.group(1))
            if match
            else -1
        )

    latest = max(
        files,
        key=get_episode,
    )

    return (
        latest,
        get_episode(latest),
    )


def save_q_checkpoint(
    q_table,
    epsilon,
    completed_episodes,
):
    path = (
        CHECKPOINT_DIR
        / (
            "pendulum_q_"
            f"{completed_episodes}_episodes.npz"
        )
    )

    np.savez_compressed(
        path,
        q_table=q_table,
        epsilon=np.array(
            [epsilon],
            dtype=np.float32,
        ),
        completed_episodes=np.array(
            [completed_episodes],
            dtype=np.int32,
        ),
    )

    print(
        "Checkpoint saved:",
        path,
    )


def append_log(
    episode,
    reward,
    epsilon,
):
    row = pd.DataFrame([{
        "episode": episode,
        "reward": reward,
        "epsilon": epsilon,
    }])

    if Q_LOG_PATH.exists():
        existing = pd.read_csv(
            Q_LOG_PATH
        )

        row = pd.concat(
            [
                existing,
                row,
            ],
            ignore_index=True,
        )

        row = (
            row
            .drop_duplicates(
                subset=["episode"],
                keep="last",
            )
            .sort_values(
                "episode"
            )
        )

    row.to_csv(
        Q_LOG_PATH,
        index=False,
    )


def load_log():
    if not Q_LOG_PATH.exists():
        return pd.DataFrame()

    return (
        pd.read_csv(
            Q_LOG_PATH
        )
        .drop_duplicates(
            subset=["episode"],
            keep="last",
        )
        .sort_values(
            "episode"
        )
        .reset_index(
            drop=True
        )
    )


def evaluate_q(
    q_table,
    episodes=EVAL_EPISODES,
):
    env = gym.make(
        ENV_ID
    )

    rewards = []
    lengths = []

    for episode in range(
        episodes
    ):
        observation, _ = env.reset(
            seed=(
                SEED
                + 10_000
                + episode
            )
        )

        state = discretise(
            observation
        )

        total_reward = 0.0
        total_length = 0
        terminated = False
        truncated = False

        while not (
            terminated
            or truncated
        ):
            action_index = int(
                np.argmax(
                    q_table[state]
                )
            )

            action = np.array(
                [
                    actions[
                        action_index
                    ]
                ],
                dtype=np.float32,
            )

            observation, reward, terminated, truncated, _ = (
                env.step(
                    action
                )
            )

            state = discretise(
                observation
            )

            total_reward += float(
                reward
            )

            total_length += 1

        rewards.append(
            total_reward
        )

        lengths.append(
            total_length
        )

    env.close()

    return {
        "mean_reward": float(
            np.mean(rewards)
        ),
        "std_reward": float(
            np.std(rewards)
        ),
        "median_reward": float(
            np.median(rewards)
        ),
        "mean_length": float(
            np.mean(lengths)
        ),
        "max_reward": float(
            np.max(rewards)
        ),
    }


def random_baseline():
    env = gym.make(
        ENV_ID
    )

    rewards = []
    lengths = []

    for episode in range(
        EVAL_EPISODES
    ):
        observation, _ = env.reset(
            seed=SEED + episode
        )

        total_reward = 0.0
        total_length = 0
        terminated = False
        truncated = False

        while not (
            terminated
            or truncated
        ):
            observation, reward, terminated, truncated, _ = (
                env.step(
                    env.action_space.sample()
                )
            )

            total_reward += float(
                reward
            )

            total_length += 1

        rewards.append(
            total_reward
        )

        lengths.append(
            total_length
        )

    env.close()

    return {
        "mean_reward": float(
            np.mean(rewards)
        ),
        "std_reward": float(
            np.std(rewards)
        ),
        "median_reward": float(
            np.median(rewards)
        ),
        "mean_length": float(
            np.mean(lengths)
        ),
        "max_reward": float(
            np.max(rewards)
        ),
    }


def show_video(
    path,
):
    path = Path(path)

    video_b64 = (
        base64.b64encode(
            path.read_bytes()
        )
        .decode("ascii")
    )

    ipythondisplay.display(
        ipythondisplay.HTML(
            f"""
            <video controls
                   style="height: 400px;
                          max-width: 100%;">
                <source
                    src="data:video/mp4;base64,{video_b64}"
                    type="video/mp4">
            </video>
            """
        )
    )


def record_q_video(
    q_table,
    filename,
    seed=SEED + 5000,
):
    env = gym.make(
        ENV_ID,
        render_mode="rgb_array",
    )

    observation, _ = env.reset(
        seed=seed
    )

    state = discretise(
        observation
    )

    frames = [
        env.render()
    ]

    terminated = False
    truncated = False

    while not (
        terminated
        or truncated
    ):
        action_index = int(
            np.argmax(
                q_table[state]
            )
        )

        action = np.array(
            [
                actions[
                    action_index
                ]
            ],
            dtype=np.float32,
        )

        observation, reward, terminated, truncated, _ = (
            env.step(
                action
            )
        )

        state = discretise(
            observation
        )

        frames.append(
            env.render()
        )

    env.close()

    path = (
        VIDEO_DIR
        / filename
    )

    imageio.mimsave(
        path,
        frames,
        fps=30,
    )

    return path

## 6. Random Baseline
Evaluate random torques before Q-Learning.

In [ ]:
random_result = (
    random_baseline()
)

random_result

## 7. Q-Learning Training
Resume from the latest saved Q-table when one exists.

In [ ]:
checkpoint, completed = (
    latest_q_checkpoint()
)

if checkpoint is None:
    q_table = np.zeros(
        (
            N_COS,
            N_SIN,
            N_VEL,
            N_ACTIONS,
        ),
        dtype=np.float32,
    )

    epsilon = EPSILON_START
    completed = 0

else:
    saved = np.load(
        checkpoint
    )

    q_table = saved[
        "q_table"
    ]

    epsilon = float(
        saved["epsilon"][0]
    )

    completed = int(
        saved[
            "completed_episodes"
        ][0]
    )

    print(
        "Resuming from:",
        checkpoint,
    )

env = gym.make(
    ENV_ID
)

for episode in range(
    completed,
    EPISODES,
):
    observation, _ = env.reset(
        seed=SEED + episode
    )

    state = discretise(
        observation
    )

    total_reward = 0.0
    terminated = False
    truncated = False

    while not (
        terminated
        or truncated
    ):
        if (
            np.random.random()
            < epsilon
        ):
            action_index = (
                np.random.randint(
                    N_ACTIONS
                )
            )

        else:
            action_index = int(
                np.argmax(
                    q_table[state]
                )
            )

        action = np.array(
            [
                actions[
                    action_index
                ]
            ],
            dtype=np.float32,
        )

        next_observation, reward, terminated, truncated, _ = (
            env.step(
                action
            )
        )

        next_state = discretise(
            next_observation
        )

        bootstrap = (
            0.0
            if terminated
            else (
                GAMMA
                * np.max(
                    q_table[
                        next_state
                    ]
                )
            )
        )

        target = (
            float(reward)
            + bootstrap
        )

        index = (
            state
            + (
                action_index,
            )
        )

        q_table[index] += (
            ALPHA
            * (
                target
                - q_table[index]
            )
        )

        state = next_state

        total_reward += float(
            reward
        )

    completed_episodes = (
        episode + 1
    )

    append_log(
        completed_episodes,
        total_reward,
        epsilon,
    )

    epsilon = max(
        EPSILON_END,
        epsilon
        * EPSILON_DECAY,
    )

    if (
        completed_episodes
        % LOG_INTERVAL_EPISODES
        == 0
    ):
        recent = (
            load_log()
            .tail(
                LOG_INTERVAL_EPISODES
            )
        )

        print(
            f"episode="
            f"{completed_episodes:,} | "
            f"mean_reward="
            f"{recent['reward'].mean():.2f} | "
            f"epsilon="
            f"{epsilon:.3f}"
        )

    if (
        completed_episodes
        % CHECKPOINT_INTERVAL_EPISODES
        == 0
    ):
        save_q_checkpoint(
            q_table,
            epsilon,
            completed_episodes,
        )

env.close()

np.save(
    MODEL_DIR
    / "pendulum_q_table.npy",
    q_table,
)

## 8. Learning Curve
Less negative reward indicates better Pendulum performance.

In [ ]:
q_curve = load_log()

q_curve["mean_100"] = (
    q_curve["reward"]
    .rolling(
        100,
        min_periods=1,
    )
    .mean()
)

plt.figure(
    figsize=(9, 4)
)

plt.plot(
    q_curve["episode"],
    q_curve["reward"],
    alpha=0.30,
    label="Episode reward",
)

plt.plot(
    q_curve["episode"],
    q_curve["mean_100"],
    linewidth=2,
    label="100-episode mean",
)

plt.title(
    "Pendulum Q-Learning Curve"
)
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

display(
    q_curve.tail(10)
)

## 9. Evaluation
Evaluate the trained Q-table without exploration.

In [ ]:
q_result = evaluate_q(
    q_table
)

q_result

## 10. Results
Compare Q-Learning with the random baseline.

In [ ]:
pendulum_results = (
    pd.DataFrame([
        {
            "Algorithm": "Random",
            **random_result,
        },
        {
            "Algorithm": "Q-Learning",
            **q_result,
        },
    ])
)

pendulum_results

## 11. Record and Play Video
Record the learned greedy Q-Learning policy and display the video.

In [ ]:
video_path = (
    record_q_video(
        q_table,
        "pendulum-qlearning.mp4",
    )
)

show_video(
    video_path
)